# Context-Aware Navigation — YOLOv26s Fine-Tuning

**Dataset:** Cityscapes + Roboflow + Local (22 classes, ~5,600 train images)  
**Storage:** AWS S3 `s3://object-detection-data-s3/merged-dataset/`

| Hyperparameter | Value | Why |
|---|---|---|
| Base model | yolo26s.pt | Lightweight, fast inference for edge deployment |
| Frozen layers | 10 (backbone) | Dataset is urban scenes — similar to COCO pretraining, backbone features reusable |
| Optimizer | AdamW | Faster convergence than SGD for fine-tuning, built-in weight decay |
| lr0 | 0.001 | Standard fine-tuning LR — not too high to destroy pretrained weights |
| lrf | 0.01 | Final LR = lr0 × lrf = 0.00001, slow decay to avoid underfitting |
| warmup_epochs | 3 | Gradually ramp up LR to avoid early instability |
| weight_decay | 0.0005 | Regularization — reduces overfitting on ~5k images |
| Epochs | 50 | With early stopping (patience=10), may stop earlier |
| Batch | 16 | Fits T4 16GB VRAM at imgsz=640 |
| imgsz | 640 | Standard YOLO input — balance of accuracy vs speed |
| conf | 0.25 | Low threshold — catch more objects, important for safety-critical use case |
| iou | 0.5 | Standard NMS overlap threshold |
| Augmentation | mosaic=1.0, flipud=0.1, fliplr=0.5, hsv | Standard YOLO augmentation for outdoor scenes |

---
## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Thu Jun  4 23:45:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

In [ ]:
!pip install ultralytics awscli boto3 pyyaml -q

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo setti

## 3. AWS Credentials
> AWS Console → IAM → Your user → Security credentials → Create access key

In [ ]:
import os

os.environ['AWS_ACCESS_KEY_ID']     = 'AWS_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'AWS_SECRET_ACCESS_KEY'
os.environ['AWS_DEFAULT_REGION']    = 'us-west-2'

# Verify
!aws s3 ls s3://object-detection-data-s3/merged-dataset/

                           PRE images/
                           PRE labels/
2026-06-04 21:55:35        413 data.yaml


## 4. Download Dataset from S3
> Takes ~5-10 minutes

In [ ]:
!aws s3 sync s3://object-detection-data-s3/merged-dataset/ /content/merged-dataset/

print("\n=== Dataset Counts ===")
for split in ['train', 'val', 'test']:
    imgs = len(os.listdir(f'/content/merged-dataset/images/{split}'))
    lbls = len(os.listdir(f'/content/merged-dataset/labels/{split}'))
    match = '✓' if imgs == lbls else f'✗ MISMATCH'
    print(f"  {split:5s}: {imgs} images, {lbls} labels  {match}")

Streaming output truncated to the last 5000 lines.
download: s3://object-detection-data-s3/merged-dataset/labels/train/cityscapes_hamburg_000000_047157_leftImg8bit.txt to merged-dataset/labels/train/cityscapes_hamburg_000000_047157_leftImg8bit.txt
download: s3://object-detection-data-s3/merged-dataset/labels/train/cityscapes_hamburg_000000_046619_leftImg8bit.txt to merged-dataset/labels/train/cityscapes_hamburg_000000_046619_leftImg8bit.txt
download: s3://object-detection-data-s3/merged-dataset/labels/train/cityscapes_hamburg_000000_046872_leftImg8bit.txt to merged-dataset/labels/train/cityscapes_hamburg_000000_046872_leftImg8bit.txt
download: s3://object-detection-data-s3/merged-dataset/labels/train/cityscapes_hamburg_000000_048494_leftImg8bit.txt to merged-dataset/labels/train/cityscapes_hamburg_000000_048494_leftImg8bit.txt
download: s3://object-detection-data-s3/merged-dataset/labels/train/cityscapes_hamburg_000000_044996_leftImg8bit.txt to merged-dataset/labels/train/cityscapes_ha

## 5. Fix data.yaml Path

In [ ]:
import yaml

yaml_path = '/content/merged-dataset/data.yaml'
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = '/content/merged-dataset'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

with open(yaml_path, 'r') as f:
    print(f.read())

path: /content/merged-dataset
train: images/train
val: images/val
test: images/test
nc: 22
names:
  0: person
  1: rider
  2: car
  3: truck
  4: bus
  5: on_rails
  6: motorcycle
  7: bicycle
  8: caravan
  9: trailer
  10: wall
  11: fence
  12: guard_rail
  13: pole
  14: polegroup
  15: vegetation
  16: terrain
  17: crosswalk
  18: pedestrian_stop
  19: pedestrian_walk
  20: stairs
  21: electric_scooter



## 6. Sanity Check — 1 Epoch on 10% of Data
> Runs in ~5 minutes. Verifies data pipeline, labels, and GPU work before committing to full training.

In [ ]:
from ultralytics import YOLO

print("Running sanity check (1 epoch, 10% data)...")
sanity_model = YOLO('yolo26s.pt')
sanity_model.train(
    data='/content/merged-dataset/data.yaml',
    epochs=1,
    imgsz=640,
    batch=16,
    fraction=0.1,     # use only 10% of data
    freeze=10,
    optimizer='AdamW',
    device=0,
    workers=2,
    name='sanity_check',
    project='/content/runs',
    exist_ok=True,
)
print("\nSanity check passed! Proceeding to full training.")

Running sanity check (1 epoch, 10% data)...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged-dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.1, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sanity_check, nbs=64, nms=False, opset=None, optimize=False, 

Exception in thread Thread-37 (plot_images):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py", line 824, in plot_images
    annotator.box_label(box, label, color=color)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py", line 326, in box_label
    ) if multi_points else self.draw.rectangle(box, width=self.lw, outline=color)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/ImageDraw.py", line 398, in rectangle
    self.draw.draw_rectangle(xy, ink, 0, width)
ValueError: x1 must be greater than or equal to x0


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.4s/it 29.9s
                   all        644      19392      0.167     0.0529     0.0101    0.00349

1 epochs completed in 0.037 hours.
Optimizer stripped from /content/runs/sanity_check/weights/last.pt, 20.3MB
Optimizer stripped from /content/runs/sanity_check/weights/best.pt, 20.3MB

Validating /content/runs/sanity_check/weights/best.pt...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,473,694 parameters, 0 gradients, 20.6 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 10% ━─────────── 2/21 1.0it/s 1.0s<18.4s

Exception in thread Thread-44 (plot_images):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py", line 824, in plot_images
    annotator.box_label(box, label, color=color)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py", line 326, in box_label
    ) if multi_points else self.draw.rectangle(box, width=self.lw, outline=color)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/ImageDraw.py", line 398, in rectangle
    self.draw.draw_rectangle(xy, ink, 0, width)
ValueError: x1 must be greater than or equal to x0


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.0it/s 20.2s
                   all        644      19392       0.12     0.0524     0.0101    0.00348
                person        347       2805     0.0873      0.389     0.0627       0.02
                 rider        202        447     0.0525     0.0783     0.0104    0.00321
                   car        387       3629      0.239      0.323      0.117     0.0448
                 truck         64         76          0          0   7.53e-06   7.53e-07
                   bus         61         83          0          0          0          0
            motorcycle         68        105    0.00307    0.00952   0.000112   1.44e-05
               bicycle        259        926     0.0384     0.0626    0.00796    0.00148
               caravan          5          5          1          0          0          0
               trailer         13         13          0          0     

## 7. Full Training
> **Only run this after sanity check passes**  
> T4: ~4-6 hrs | L4: ~2-3 hrs | A100: ~1-2 hrs  
> Checkpoints saved every 5 epochs to `/content/runs/navigation_model/weights/`

In [ ]:
from ultralytics import YOLO
import subprocess

# ── Auto-save to S3 every 5 epochs ──────────────────────────────
def save_to_s3(trainer):
    if trainer.epoch % 5 == 0:
        print(f"\n[Epoch {trainer.epoch}] Saving checkpoint to S3...")
        subprocess.run([
            'aws', 's3', 'sync',
            '/content/runs/navigation_model/',
            's3://object-detection-data-s3/training-runs/navigation_model/'
        ])
        print("[Saved to S3]")
# ────────────────────────────────────────────────────────────────

model = YOLO('yolo26s.pt')
model.add_callback('on_train_epoch_end', save_to_s3)

results = model.train(
    # ── Data ────────────────────────────────
    data       = '/content/merged-dataset/data.yaml',
    imgsz      = 640,
    batch      = 32,

    # ── Training ────────────────────────────
    epochs     = 50,
    freeze     = 10,          # freeze backbone (layers 0-9)
    optimizer  = 'AdamW',
    lr0        = 0.001,       # initial learning rate
    lrf        = 0.01,        # final lr = lr0 * lrf
    weight_decay = 0.0005,
    warmup_epochs = 3,
    patience   = 10,          # early stopping


    # ── Augmentation ────────────────────────
    mosaic     = 0.5,         # mosaic augmentation (combines 4 images)
    fliplr     = 0.5,         # horizontal flip 50% chance
    flipud     = 0.1,         # vertical flip 10% (rare in street scenes)
    hsv_h      = 0.015,       # hue variation (lighting changes)
    hsv_s      = 0.7,         # saturation variation
    hsv_v      = 0.4,         # brightness variation (day/night)
    mixup      = 0.0,
    copy_paste = 0.0,

    # ── Validation ──────────────────────────
    val        = True,        # validate every epoch
    conf       = 0.25,        # confidence threshold
    iou        = 0.5,         # NMS IoU threshold

    # ── Saving ──────────────────────────────
    save_period = 5,          # checkpoint every 5 epochs
    name       = 'navigation_model',
    project    = '/content/runs',
    exist_ok   = True,
    plots = False,

    # ── System ──────────────────────────────
    device     = 0,
    workers    = 2,
)

print("\nTraining complete!")
print(f"Best mAP50: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=0.25, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged-dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.5, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=navigation_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pa

## 8. Save Final Results to S3

In [ ]:
!aws s3 sync /content/runs/ s3://object-detection-data-s3/training-runs/
print("All results saved to S3!")

upload: runs/detect/val/BoxF1_curve.png to s3://object-detection-data-s3/training-runs/detect/val/BoxF1_curve.png
upload: runs/detect/val/val_batch2_labels.jpg to s3://object-detection-data-s3/training-runs/detect/val/val_batch2_labels.jpg
upload: runs/detect/val/val_batch2_pred.jpg to s3://object-detection-data-s3/training-runs/detect/val/val_batch2_pred.jpg
upload: runs/detect/val/confusion_matrix.png to s3://object-detection-data-s3/training-runs/detect/val/confusion_matrix.png
upload: runs/detect/val/BoxPR_curve.png to s3://object-detection-data-s3/training-runs/detect/val/BoxPR_curve.png
upload: runs/navigation_model/results.csv to s3://object-detection-data-s3/training-runs/navigation_model/results.csv
upload: runs/detect/val/val_batch0_pred.jpg to s3://object-detection-data-s3/training-runs/detect/val/val_batch0_pred.jpg
upload: runs/detect/val/BoxP_curve.png to s3://object-detection-data-s3/training-runs/detect/val/BoxP_curve.png
upload: runs/detect/val/val_batch1_pred.jpg to s

## 9. Evaluate on Test Set

In [ ]:
from ultralytics import YOLO

best_model = YOLO('/content/runs/navigation_model/weights/best.pt')

metrics = best_model.val(
    data  = '/content/merged-dataset/data.yaml',
    split = 'test',
    conf  = 0.25,
    iou   = 0.5,
)

print("\n=== Overall Test Results ===")
print(f"  mAP50:     {metrics.box.map50:.4f}")
print(f"  mAP50-95:  {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall:    {metrics.box.mr:.4f}")

print("\n=== Per-Class Results ===")
class_names = [
    'person','rider','car','truck','bus','on_rails','motorcycle','bicycle',
    'caravan','trailer','wall','fence','guard_rail','pole','polegroup',
    'vegetation','terrain','crosswalk','pedestrian_stop','pedestrian_walk',
    'stairs','electric_scooter'
]
print(f"  {'Class':<20} {'mAP50':>8} {'mAP50-95':>10}")
print("  " + "-" * 42)
for i, (ap50, ap) in enumerate(zip(metrics.box.ap50, metrics.box.ap)):
    print(f"  {class_names[i]:<20} {ap50:>8.4f} {ap:>10.4f}")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,473,694 parameters, 0 gradients, 20.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2218.4±1376.0 MB/s, size: 1346.9 KB)
val: Scanning /content/merged-dataset/labels/test.cache... 167 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 167/167 50.0Mit/s 0.0s
val: /content/merged-dataset/images/test/cityscapes_frankfurt_000001_028232_leftImg8bit.png: 1 duplicate labels removed
val: /content/merged-dataset/images/test/cityscapes_frankfurt_000001_046272_leftImg8bit.png: 1 duplicate labels removed
val: /content/merged-dataset/images/test/cityscapes_munster_000026_000019_leftImg8bit.png: 2 duplicate labels removed
val: /content/merged-dataset/images/test/cityscapes_munster_000141_000019_leftImg8bit.png: 1 duplicate labels removed
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 5238, len(boxes) = 5249. To resolve this only boxes will 

## 10. Resume Training if Colab Disconnects
> Uncomment and run all cells below if session resets

In [ ]:
# Step 1 — Re-download data and checkpoints from S3
# !aws s3 sync s3://object-detection-data-s3/merged-dataset/ /content/merged-dataset/
# !aws s3 sync s3://object-detection-data-s3/training-runs/ /content/runs/

# Step 2 — Resume from last checkpoint
# from ultralytics import YOLO
# model = YOLO('/content/runs/navigation_model/weights/last.pt')
# model.train(resume=True)